# 3 · Searching the space

The small space can be simulated exhaustively, larger ones cannot. Here the search gets a budget of simulations, and we compare what it finds with the optimum we know from simulating everything.

In [ ]:
# This is just the code header, ignore it.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import dse
import dse.plots as plots
from dse import DATA, WORK

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

In [ ]:
mlir = DATA / "small" / "gemv.mlir"
oracle = dse.load_pool(DATA / "small" / "pool.csv")
best = oracle["cost"].min()

# Random search needs no simulator once the oracle exists; 2000 trials of it
# give the reference every search below is measured against.
baseline = dse.random_search_baseline(oracle, budget=100)

## 40 simulations

Cinnamon searches with Bayesian optimisation. After a random initial sample, a surrogate model predicts the latency (µ) and the uncertainty of that prediction (σ) for the configurations not simulated yet. The next configuration to simulate is chosen from these predictions.

In [ ]:
r40 = dse.bo(mlir, "bo40", max_evals=40, n_init=10)
pool40 = r40.pool()
v = dse.visited(pool40)

print(f"{len(v)} simulations in {r40.seconds:.1f} s")
print(f"best found {v['cost'].min():.3f} ms — rank {dse.rank_of(v['cost'].min(), oracle)} of {len(oracle)}")

The configurations the search simulated, in order. The first 10 are the random initial sample. For the others, `mu` and `sigma` are what the surrogate predicted before they were simulated, and `acq` is the score that selected them.

In [ ]:
cols = dse.param_columns(pool40) + ["cost", "mu", "sigma", "acq"]
v[cols].style.format({"cost": "{:.2f}", "mu": "{:.2f}", "sigma": "{:.2f}", "acq": "{:.2f}"}).hide(axis="index")

In [ ]:
plots.surrogate_check(pool40)
plt.show()

Points on the dotted line were predicted correctly. The bars show the predicted uncertainty.

In [ ]:
ax = plots.best_so_far({"Bayesian, 40 evals": pool40}, oracle_best=best,
                       baseline=baseline[:, :40], n_init=10)
plt.show()

## Several seeds

The search is randomised. Running it from 5 seeds shows how much the result depends on the seed.

In [ ]:
r5 = dse.bo(mlir, "bo40x5", max_evals=40, n_init=10, seeds=5)
pools5 = r5.pools()

print(f"5 × 40 simulations in {r5.seconds:.0f} s")
for seed, p in sorted(pools5.items()):
    m = dse.visited(p)["cost"].min()
    print(f"seed {seed:4d}: best {m:6.3f} ms  rank {dse.rank_of(m, oracle):4d}  ({(m / best - 1) * 100:4.0f} % above optimum)")

In [ ]:
ax = plots.seeds_spread(pools5, oracle_best=best, baseline=baseline[:, :40])
plt.show()

With 40 simulations, some seeds find the best configurations and some do not.

## 100 simulations

In [ ]:
r100 = dse.bo(mlir, "bo100", max_evals=100, n_init=20)
pool100 = r100.pool()
m = dse.visited(pool100)["cost"].min()
print(f"100 simulations in {r100.seconds:.1f} s — best {m:.3f} ms, rank {dse.rank_of(m, oracle)}")

ax = plots.best_so_far({"Bayesian, 40 evals": pool40, "Bayesian, 100 evals": pool100},
                       oracle_best=best, baseline=baseline, n_init=20)
plt.show()

With a larger budget the search finds the optimum more often. Random search with the same budget (dashed) stays well above it.

## A larger space

4096 × 4096 on up to 2048 DPUs, so the number of DPUs is a parameter too. We simulated its 26,238 feasible configurations in advance (7 minutes on 14 cores).

In [ ]:
big_mlir = DATA / "big" / "gemv.mlir"
big_oracle = dse.load_pool(DATA / "big" / "pool.csv")
big_space = dse.load_space(DATA / "big" / "space.json")
big_best = big_oracle["cost"].min()

print(dse.cinm.describe_space(big_space))
print()
print(f"best {big_best:.2f} ms, median {big_oracle['cost'].median():.1f} ms; "
      f"within 10 % of the best: {(big_oracle['cost'] <= 1.1 * big_best).sum()} configurations")

In [ ]:
rb = dse.bo(big_mlir, "big_bo", max_evals=100, n_init=20)
poolb = rb.pool()
mb = dse.visited(poolb)["cost"].min()
big_baseline = dse.random_search_baseline(big_oracle, budget=100)

print(f"100 simulations in {rb.seconds:.1f} s")
print(f"best found {mb:.2f} ms — rank {dse.rank_of(mb, big_oracle)} of {len(big_oracle)}, "
      f"{(mb / big_best - 1) * 100:.0f} % above the optimum")
print(f"random search, same budget: median {np.median(big_baseline[:, -1]):.1f} ms "
      f"({(np.median(big_baseline[:, -1]) / big_best - 1) * 100:.0f} % above)")

In [ ]:
ax = plots.best_so_far({"Bayesian, 100 evals": poolb}, oracle_best=big_best,
                       baseline=big_baseline, n_init=20,
                       title="26,238 feasible configurations — best latency found vs. simulations")
plt.show()

In [ ]:
cols = dse.param_columns(big_oracle) + ["cost"]
big_oracle.sort_values("cost").head(5)[cols].style.format({"cost": "{:.3f}"}).hide(axis="index")

Here the best configuration uses 1 tasklet on 2048 DPUs. For the small problem it was 8 tasklets on 64 DPUs.

## Extra: gemm

A matrix–matrix product: 9 parameters and 2.8 million feasible configurations. We have no oracle for this one.

In [ ]:
gemm_mlir = DATA / "large" / "gemm.mlir"
gemm_space = dse.dump_space(gemm_mlir, "gemm_space")
print(dse.cinm.describe_space(gemm_space))
print(f"\nAt 60 ms per simulation, enumerating it would take "
      f"{gemm_space['feasible_size'] * 0.06 / 3600:.0f} hours on one core.")

In [ ]:
rl = dse.bo(gemm_mlir, "gemm_bo", max_evals=60, n_init=15)
pooll = rl.pool()
vl = dse.visited(pooll)

print(f"60 simulations in {rl.seconds:.1f} s")
ax = plots.best_so_far({"Bayesian, 60 evals": pooll}, n_init=15,
                       title="2.8 million feasible configurations — no oracle to compare with")
plt.show()

vl.sort_values("cost").head(3)[dse.param_columns(pooll) + ["cost"]].style.format({"cost": "{:.2f}"}).hide(axis="index")

Without an oracle we cannot tell how close this is to the optimum. The two previous spaces show how well the search does with a similar budget when we can check.